# Analiza Cererii de Rambursare a Cheltuielilor

Acest caiet demonstrează cum să creezi agenți care utilizează plugin-uri pentru a procesa cheltuielile de călătorie din imagini locale ale chitanțelor, să genereze un e-mail de cerere de rambursare a cheltuielilor și să vizualizeze datele despre cheltuieli folosind un grafic circular. Agenții aleg dinamic funcțiile în funcție de contextul sarcinii.

Pași:
1. Agentul OCR procesează imaginea locală a chitanței și extrage datele despre cheltuielile de călătorie.
2. Agentul de e-mail generează un e-mail de cerere de rambursare a cheltuielilor.

### Exemplu de scenariu de cheltuieli de călătorie:
Imaginează-ți că ești un angajat care călătorește pentru o întâlnire de afaceri într-un alt oraș. Compania ta are o politică de rambursare a tuturor cheltuielilor rezonabile legate de călătorie. Iată o defalcare a potențialelor cheltuieli de călătorie:
- Transport:
Bilet de avion dus-întors între orașul tău de origine și orașul destinație.
Taxi sau servicii de ride-hailing către și de la aeroport.
Transport local în orașul destinație (precum transportul public, mașini închirieri sau taxiuri).

- Cazare:
Ședere la hotel pentru trei nopți într-un hotel de afaceri de nivel mediu aproape de locul întâlnirii.

- Mese:
Alocație zilnică pentru micul dejun, prânz și cină, bazată pe politica per diem a companiei.

- Cheltuieli Diverse:
Taxe de parcare la aeroport.
Costuri pentru acces la internet în hotel.
Bacșișuri sau mici taxe de serviciu.

- Documentație:
Îți predai toate chitanțele (zboruri, taxiuri, hotel, mese etc.) și un raport complet al cheltuielilor pentru rambursare.


## Importați bibliotecile necesare

Importați bibliotecile și modulele necesare pentru caiet.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Define Modele de Cheltuieli

 Creați un model Pydantic pentru cheltuielile individuale și o clasă ExpenseFormatter pentru a converti o interogare utilizator într-un set structurat de date despre cheltuieli.

 Fiecare cheltuială va fi reprezentată în formatul:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definirea instrumentelor - Generarea emailului

Creați o funcție instrument pentru generarea unui email pentru trimiterea unei cereri de rambursare a cheltuielilor.
- Acest instrument folosește decoratorul `@tool` din Microsoft Agent Framework.
- Calculează suma totală a cheltuielilor și formatează detaliile într-un corp de email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Unealtă pentru extragerea cheltuielilor de călătorie din imaginile chitanțelor

Creează o funcție unealtă pentru extragerea cheltuielilor de călătorie din imaginile chitanțelor.
- Această unealtă folosește decoratorul `@tool` din Microsoft Agent Framework.
- Citește imaginea chitanței, o codifică în base64 și returnează URI-ul de date pentru ca agentul să îl analizeze.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Procesarea Cheltuielilor

Definiți agenții și conectați-i într-un flux de lucru secvențial folosind `WorkflowBuilder`.
- Agentul OCR extrage date structurate despre cheltuieli din imaginea bonului folosind instrumentul `load_receipt_image`.
- Agentul de e-mail preia datele extrase și generează un email profesional pentru cererea de decontare a cheltuielilor folosind instrumentul `generate_expense_email`.
- `WorkflowBuilder` cu `add_edge` creează o succesiune secvențială: Agent OCR → Agent e-mail.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Funcția principală

Construiți fluxul de lucru secvențial și rulați-l pentru a procesa imaginea chitanței și a genera emailul pentru cererea de despăgubire.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertisment**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să aveți în vedere că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un traducător uman. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări eronate cauzate de utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
